# Training Techniques

The difference between a model that works and one that doesn't is rarely the architecture — it's how you train it. This notebook covers the techniques that practitioners use daily:

- Diagnosing overfitting/underfitting
- Regularization (dropout, batch norm)
- Weight initialization
- Learning rate scheduling
- Optimizer selection
- Early stopping
- Transfer learning

We'll use the same model on the same dataset for every comparison so differences are due to the technique, not randomness.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
X_np, y_np = make_moons(n_samples=300, noise=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_np, y_np, test_size=0.3, random_state=42)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.FloatTensor(y_val).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

def train_model(model, optimizer, epochs=200, verbose=False):
    criterion = nn.BCELoss()
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            pred = model(X_b)
            loss = criterion(pred, y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))
        
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t.to(device))
            val_loss = criterion(val_pred, y_val_t.to(device))
            val_losses.append(val_loss.item())
    
    return train_losses, val_losses

print(f"Train: {len(X_train)}, Validation: {len(X_val)}")

---
## 1. Overfitting vs Underfitting

The most fundamental diagnostic tool: **learning curves**.

- **Overfitting**: train loss keeps dropping, val loss starts rising → model memorizes training data
- **Underfitting**: both losses stay high → model is too simple
- **Good fit**: both losses converge to similar low values

In [ ]:
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

class HugeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

class RightModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
configs = [
    (TinyModel, 'Underfitting (too simple)', '#e74c3c'),
    (HugeModel, 'Overfitting (too complex)', '#f39c12'),
    (RightModel, 'Good Fit (just right)', '#2ecc71'),
]

for ax, (ModelClass, title, color) in zip(axes, configs):
    torch.manual_seed(42)
    m = ModelClass().to(device)
    opt = optim.Adam(m.parameters(), lr=0.01)
    tl, vl = train_model(m, opt, epochs=300)
    
    ax.plot(tl, label='Train', color=color, linewidth=2)
    ax.plot(vl, label='Validation', color=color, linewidth=2, linestyle='--')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(tl[0], vl[0]) * 1.1)

plt.tight_layout()
plt.show()
print("Diagnosis: look at the GAP between train and val loss.")
print("Big gap = overfitting. Both high = underfitting. Small gap, both low = good.")

---
## 2. Dropout — Random Neuron Deactivation

During training, randomly set a fraction of neurons to zero. This forces the network to not rely on any single neuron — a form of ensemble learning within one model.

At inference, all neurons are active but outputs are scaled by the dropout rate.

In [ ]:
class ModelNoDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

class ModelWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, ModelClass, title in [(ax1, ModelNoDropout, 'Without Dropout'),
                               (ax2, ModelWithDropout, 'With Dropout (p=0.3)')]:
    torch.manual_seed(42)
    m = ModelClass().to(device)
    opt = optim.Adam(m.parameters(), lr=0.01)
    tl, vl = train_model(m, opt, epochs=300)
    
    ax.plot(tl, label='Train', color='#3498db', linewidth=2)
    ax.plot(vl, label='Validation', color='#e74c3c', linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Dropout narrows the gap between train and val loss → less overfitting.")

---
## 3. Batch Normalization

Normalizes activations within each mini-batch to have mean≈0 and std≈1. Benefits:
- **Faster training**: higher learning rates work
- **Less sensitive to weight initialization**
- **Slight regularization** (noise from batch statistics)

In [ ]:
class ModelNoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

class ModelWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

fig, ax = plt.subplots(figsize=(10, 5))

for ModelClass, label, color in [(ModelNoBN, 'Without BatchNorm', '#e74c3c'),
                                  (ModelWithBN, 'With BatchNorm', '#2ecc71')]:
    torch.manual_seed(42)
    m = ModelClass().to(device)
    opt = optim.Adam(m.parameters(), lr=0.01)
    tl, vl = train_model(m, opt, epochs=200)
    ax.plot(tl, label=f'{label} (train)', color=color, linewidth=2)
    ax.plot(vl, label=f'{label} (val)', color=color, linewidth=2, linestyle='--')

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Effect of Batch Normalization', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Weight Initialization

Bad initialization can make training impossible. If all weights are the same, all neurons learn the same thing. If weights are too large, activations saturate. Too small, and the signal dies.

| Method | Formula | Best for |
|--------|---------|----------|
| Xavier/Glorot | \\(W \sim N(0, \frac{2}{n_{in}+n_{out}})\\) | Sigmoid, Tanh |
| He/Kaiming | \\(W \sim N(0, \frac{2}{n_{in}})\\) | ReLU |

In [ ]:
def create_model_with_init(init_fn):
    m = nn.Sequential(
        nn.Linear(2, 32), nn.ReLU(),
        nn.Linear(32, 32), nn.ReLU(),
        nn.Linear(32, 32), nn.ReLU(),
        nn.Linear(32, 32), nn.ReLU(),
        nn.Linear(32, 1), nn.Sigmoid()
    )
    for layer in m:
        if isinstance(layer, nn.Linear):
            init_fn(layer.weight)
            nn.init.zeros_(layer.bias)
    return m.to(device)

inits = {
    'Random N(0,1)': lambda w: nn.init.normal_(w, std=1.0),
    'Random N(0,0.01)': lambda w: nn.init.normal_(w, std=0.01),
    'Xavier': nn.init.xavier_normal_,
    'He/Kaiming': lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'),
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

for (name, init_fn), color in zip(inits.items(), colors):
    torch.manual_seed(42)
    m = create_model_with_init(init_fn)
    opt = optim.Adam(m.parameters(), lr=0.01)
    tl, _ = train_model(m, opt, epochs=200)
    ax.plot(tl, label=name, color=color, linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Impact of Weight Initialization', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("He initialization converges fastest for ReLU networks.")
print("Too-large weights (N(0,1)) can cause initial instability.")
print("Too-small weights (N(0,0.01)) converge slowly — weak signal.")

---
## 5. Learning Rate Scheduling

Start with a larger learning rate (fast progress) and reduce it over time (fine-tune). Most common strategies:

In [ ]:
epochs = 100
dummy_model = nn.Linear(10, 1)

schedulers = {
    'StepLR (γ=0.5 every 30)': lambda opt: optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.5),
    'CosineAnnealing': lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs),
    'ReduceOnPlateau': lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5),
    'ExponentialLR (γ=0.97)': lambda opt: optim.lr_scheduler.ExponentialLR(opt, gamma=0.97),
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for (name, sched_fn), color in zip(schedulers.items(), colors):
    lrs = []
    opt = optim.SGD([nn.Parameter(torch.randn(1))], lr=0.1)
    scheduler = sched_fn(opt)
    
    for epoch in range(epochs):
        lrs.append(opt.param_groups[0]['lr'])
        if 'Plateau' in name:
            fake_loss = 1.0 / (epoch + 1) + np.random.random() * 0.05
            scheduler.step(fake_loss)
        else:
            scheduler.step()
    
    ax.plot(lrs, label=name, color=color, linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Scheduling Strategies', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("StepLR:          Simple — cut LR by half every N epochs")
print("CosineAnnealing: Smooth decay — popular in practice")
print("ReduceOnPlateau: Adaptive — reduces when metric stops improving")
print("ExponentialLR:   Aggressive — continuous exponential decay")

---
## 6. Optimizers Compared

Training the exact same model with different optimizers on the same data.

In [ ]:
def make_model():
    return nn.Sequential(
        nn.Linear(2, 32), nn.ReLU(),
        nn.Linear(32, 16), nn.ReLU(),
        nn.Linear(16, 1), nn.Sigmoid()
    ).to(device)

optimizer_configs = {
    'SGD (lr=0.1)': lambda m: optim.SGD(m.parameters(), lr=0.1),
    'SGD + Momentum': lambda m: optim.SGD(m.parameters(), lr=0.1, momentum=0.9),
    'Adam (lr=0.01)': lambda m: optim.Adam(m.parameters(), lr=0.01),
    'AdamW (lr=0.01)': lambda m: optim.AdamW(m.parameters(), lr=0.01, weight_decay=0.01),
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

for (name, opt_fn), color in zip(optimizer_configs.items(), colors):
    torch.manual_seed(42)
    m = make_model()
    opt = opt_fn(m)
    tl, _ = train_model(m, opt, epochs=200)
    ax.plot(tl, label=name, color=color, linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Optimizer Comparison — Same Model, Same Data', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("SGD:        Simplest. Can be slow but finds flat minima (better generalization).")
print("SGD+Mom:    Adds velocity — accelerates through flat regions, dampens oscillations.")
print("Adam:       Adaptive per-parameter LR. Fast convergence. The default choice.")
print("AdamW:      Adam with proper weight decay. Use for fine-tuning pretrained models.")

---
## 7. Early Stopping

Stop training when validation loss stops improving. This prevents overfitting and saves compute.

Implementation: track the best validation loss. If it hasn't improved for `patience` epochs, stop.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_weights = None
    
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

torch.manual_seed(42)
model_es = HugeModel().to(device)
optimizer = optim.Adam(model_es.parameters(), lr=0.01)
criterion = nn.BCELoss()
early_stop = EarlyStopping(patience=20)

train_losses, val_losses = [], []
stopped_at = 500

for epoch in range(500):
    model_es.train()
    epoch_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        pred = model_es(X_b)
        loss = criterion(pred, y_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))
    
    model_es.eval()
    with torch.no_grad():
        val_pred = model_es(X_val_t.to(device))
        val_loss = criterion(val_pred, y_val_t.to(device)).item()
        val_losses.append(val_loss)
    
    if early_stop(val_loss, model_es):
        stopped_at = epoch + 1
        model_es.load_state_dict(early_stop.best_weights)
        break

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train', color='#3498db', linewidth=2)
ax.plot(val_losses, label='Validation', color='#e74c3c', linewidth=2)
ax.axvline(stopped_at - early_stop.patience - 1, color='#2ecc71', linestyle='--',
           linewidth=2, label=f'Best model (epoch {stopped_at - early_stop.patience})')
ax.axvline(stopped_at - 1, color='gray', linestyle=':', linewidth=2, label=f'Stopped (epoch {stopped_at})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Early Stopping in Action', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Stopped training at epoch {stopped_at} (patience={early_stop.patience})")
print(f"Best validation loss: {early_stop.best_loss:.4f}")
print(f"Restored best weights — model is from the point of lowest val loss.")

---
## 8. Transfer Learning Preview

Why train a feature extractor from scratch when models pretrained on ImageNet (1.2M images, 1000 classes) already learned excellent features?

**Transfer learning strategy:**
1. Take a pretrained model (e.g., ResNet, VGG)
2. Freeze the feature extraction layers
3. Replace the final classification layer for your task
4. Train only the new layers (or fine-tune everything with a small LR)

In [ ]:
from torchvision import models

resnet = models.resnet18(weights='IMAGENET1K_V1')

print("ResNet-18 architecture (last few layers):")
children = list(resnet.children())
for i, child in enumerate(children[-3:]):
    print(f"  Layer {i + len(children) - 3}: {child.__class__.__name__}")

print(f"\nOriginal final layer: {resnet.fc}")
print(f"  → 1000 classes (ImageNet)")

for param in resnet.parameters():
    param.requires_grad = False

num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 10)

print(f"\nNew final layer: {resnet.fc}")
print(f"  → 10 classes (our task)")

total_params = sum(p.numel() for p in resnet.parameters())
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")
print(f"\nWe only train {trainable_params/total_params:.1%} of the model — the rest uses ImageNet features!")

---
## Quick Reference — When Things Go Wrong

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Loss not decreasing | LR too low or bad init | Increase LR, use He init |
| Loss oscillating wildly | LR too high | Reduce LR by 10x |
| Loss = NaN | LR way too high, or numerical issue | Reduce LR, add gradient clipping |
| Train loss good, val loss bad | Overfitting | Add dropout, reduce model size, early stopping |
| Both losses stay high | Underfitting | Bigger model, train longer, lower regularization |
| Very slow training | LR too low or no BatchNorm | Increase LR, add BatchNorm |
| Accuracy stuck at random | Wrong loss function or data issue | Check labels, use correct loss |
| Gradients all zero | Dead ReLU or vanishing gradient | Use He init, try LeakyReLU |

---
## Key Takeaways

1. **Learning curves** are your primary diagnostic tool — always plot train vs val loss
2. **Dropout** prevents co-adaptation of neurons → reduces overfitting
3. **Batch Normalization** normalizes layer inputs → faster, more stable training
4. **He initialization** for ReLU, Xavier for sigmoid/tanh
5. **Cosine annealing** or **ReduceOnPlateau** for learning rate scheduling
6. **Adam** is the safe default optimizer; **SGD+momentum** for state-of-the-art results
7. **Early stopping** prevents overfitting and saves compute
8. **Transfer learning** gives you ImageNet-quality features for free